In [4]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates # For better date axis formatting

# Local address of the Fission router (via port-forwarding)
FISSION_ROUTER_URL = "http://localhost:9090"

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def fetch_sentiment_data(keyword, start_date, end_date, source=None):
    """
    Calls the ui-sentiment Fission function to fetch sentiment data.
    Optionally, specify 'source' to extract data for a specific source only.
    """
    # URL encode the keyword to handle special characters (e.g., spaces or wildcard '*')
    # If keyword is None or empty, we can use a default wildcard representation, e.g., "*"
    keyword_for_url = "*" if not keyword else requests.utils.quote(keyword)

    api_url = f"{FISSION_ROUTER_URL}/ui/sentiment/keyword/{keyword_for_url}/start/{start_date}/end/{end_date}"
    
    print(f"Calling API: {api_url}")
    
    try:
        # It's recommended to set a longer timeout for Fission function calls
        response = requests.get(api_url, timeout=600) # e.g., 10 minutes
        response.raise_for_status() # Raise an exception if HTTP status code indicates an error
        
        all_sentiment_data = response.json()
        print("Successfully fetched sentiment data from API!")

        if source:
            if source in all_sentiment_data:
                return all_sentiment_data[source]
            else:
                print(f"Warning: Specified source '{source}' not found in API response. Available sources: {list(all_sentiment_data.keys())}")
                return None
        else:
            # If no source is specified, return data for all sources
            return all_sentiment_data
            
    except requests.exceptions.Timeout:
        print(f"Error: API request timed out.")
        return None
    except requests.exceptions.HTTPError as http_err:
        print(f"API HTTP Error: {http_err}")
        print(f"Response Status Code: {response.status_code}")
        print(f"Response Content: {response.text}")
        return None
    except requests.exceptions.RequestException as req_err:
        print(f"Error during API request: {req_err}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not parse JSON response from API. Response content: {response.text}")
        return None


In [ ]:
def plot_sentiment_data(sentiment_data_dict, title_prefix=""):
    """
    Converts a sentiment data dictionary (by date) to a Pandas DataFrame and plots sentiment trends.
    The format of sentiment_data_dict should be: { "YYYY-MM-DD": {"neg": N, "neu": N, "pos": N, "compound": N, "count": N}, ... }
    """
    if not sentiment_data_dict or not isinstance(sentiment_data_dict, dict):
        print(f"{title_prefix}: No valid sentiment data provided for plotting.")
        return

    try:
        df = pd.DataFrame.from_dict(sentiment_data_dict, orient='index')
        if df.empty:
            print(f"{title_prefix}: Sentiment data is empty, cannot plot.")
            return

        df.index = pd.to_datetime(df.index) # Convert index to datetime objects
        df = df.sort_index() # Sort by date

        print(f"\n{title_prefix} Sentiment Data Table:")
        print(df)

        # Plot daily average compound score
        if 'compound' in df.columns:
            plt.figure(figsize=(15, 7))
            
            # If 'count' column exists and you want to plot average (assuming 'compound' is sum)
            # if 'count' in df.columns:
            #     df['avg_compound'] = df.apply(lambda row: row['compound'] / row['count'] if row['count'] > 0 else 0, axis=1)
            #     plt.plot(df.index, df['avg_compound'], marker='o', linestyle='-', label='Daily Average Compound')
            # else:
            #     plt.plot(df.index, df['compound'], marker='o', linestyle='-', label='Daily Compound')

            # Directly plot the 'compound' column (assuming it's the desired daily representative value, e.g., average or sum)
            plt.plot(df.index, df['compound'], marker='o', linestyle='-', label='Daily Compound Score')
            
            # Plot positive and negative sentiment scores
            if 'pos' in df.columns:
                plt.plot(df.index, df['pos'], marker='^', linestyle='--', color='green', alpha=0.7, label='Daily Positive Sentiment')
            if 'neg' in df.columns:
                plt.plot(df.index, df['neg'], marker='v', linestyle='--', color='red', alpha=0.7, label='Daily Negative Sentiment')


            plt.title(f"{title_prefix} Keyword Daily Sentiment Trend")
            plt.xlabel("Date")
            plt.ylabel("Sentiment Score")
            plt.grid(True, which='both', linestyle='--', linewidth=0.5)
            plt.axhline(0, color='black', linestyle='-', linewidth=0.8) # Mark the neutral line
            
            # Format X-axis date display
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=10)) # Automatically select appropriate date ticks
            plt.xticks(rotation=45) # Rotate date labels to prevent overlap
            
            plt.legend()
            plt.tight_layout() # Automatically adjust subplot parameters for a tight layout
            plt.show()
        else:
            print(f"{title_prefix}: 'compound' column missing from data, cannot plot Compound score trend.")

    except Exception as e:
        print(f"{title_prefix}: Error processing or plotting sentiment data: {e}")
        import traceback
        traceback.print_exc()


In [ ]:
keyword_to_analyze = "*"       # Keyword you want to analyze, can be "*" for all, or None/empty string
analysis_start_date = "2024-01-01"  # Start date (YYYY-MM-DD)
analysis_end_date = "2025-05-19"    # End date (YYYY-MM-DD)

# Fetch sentiment data for all sources
all_sources_data = fetch_sentiment_data(keyword_to_analyze, analysis_start_date, analysis_end_date)

if all_sources_data:
    # Process and plot Reddit sentiment data separately
    if "reddit" in all_sources_data:
        plot_sentiment_data(all_sources_data["reddit"], title_prefix=f"Reddit '{keyword_to_analyze}'")
    else:
        print("Reddit sentiment data not found in API response.")

else:
    print("Failed to fetch any sentiment data from the API. Please check Fission function logs and service status.")